In [1]:
import sys
import torch
sys.path.insert(0, r"d:\VS codes\ML Coding")

from model.detector import build_detector, get_model_summary

NUM_CLASSES = 7   # background + person + bicycle + car + motorcycle + bus + truck
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_detector(
    num_classes         = NUM_CLASSES,
    backbone_name       = "resnet50",
    pretrained_backbone = True
)

model = model.to(device)
print(f"Model on device: {device}")

[Backbone] Built resnet50 + FPN
[Backbone] Output channels: 256
[Backbone] Pretrained: True
[Backbone] Frozen params: 24 | Trainable params: 45

[Detector] Model built successfully
[Detector] Backbone    : resnet50 + FPN
[Detector] Num classes : 7 (including background)
[Detector] Anchors     : 5 levels x 3 ratios = 15 anchor types
[Detector] ROI output  : 7x7 per proposal

Model on device: cpu


In [2]:
get_model_summary(model)


========= Model Summary =========
  Component              Total    Trainable
  ----------------------------------------
  Backbone          26,799,296   25,364,480
  RPN                  593,935      593,935
  ROI Head          13,931,555   13,931,555
  ----------------------------------------
  TOTAL             41,324,786   39,889,970



In [3]:
# In training mode the model expects images AND targets
# It returns a dict of losses

model.train()

# Simulate a batch of 2 images
dummy_images = [
    torch.rand(3, 800, 800).to(device),
    torch.rand(3, 600, 800).to(device),
]

# Simulate targets (what we'd get from our DataLoader)
dummy_targets = [
    {
        "boxes":    torch.tensor([[50., 30., 200., 250.],
                                  [300., 100., 500., 400.]], device=device),
        "labels":   torch.tensor([1, 3], device=device),  # person, car
        "image_id": torch.tensor([1],    device=device),
        "area":     torch.tensor([24500., 60000.], device=device),
        "iscrowd":  torch.tensor([0, 0], device=device),
    },
    {
        "boxes":    torch.tensor([[10., 20., 100., 150.]], device=device),
        "labels":   torch.tensor([1], device=device),     # person
        "image_id": torch.tensor([2], device=device),
        "area":     torch.tensor([13000.], device=device),
        "iscrowd":  torch.tensor([0], device=device),
    },
]

with torch.no_grad():
    loss_dict = model(dummy_images, dummy_targets)

print("=== Training Mode Output (Loss Dict) ===\n")
total_loss = 0
for name, value in loss_dict.items():
    print(f"  {name:<25} : {value.item():.4f}")
    total_loss += value.item()
print(f"\n  {'Total loss':<25} : {total_loss:.4f}")
print("\nAll 4 losses present — model training forward pass works!")

=== Training Mode Output (Loss Dict) ===

  loss_classifier           : 2.1417
  loss_box_reg              : 0.0104
  loss_objectness           : 0.7021
  loss_rpn_box_reg          : 0.0063

  Total loss                : 2.8606

All 4 losses present — model training forward pass works!


In [4]:
# In eval mode the model only needs images
# It returns detections — no targets needed

model.eval()

dummy_images = [
    torch.rand(3, 800, 800).to(device),
]

with torch.no_grad():
    detections = model(dummy_images)

print("=== Inference Mode Output ===\n")
det = detections[0]
print(f"  Keys in output  : {list(det.keys())}")
print(f"  Boxes shape     : {det['boxes'].shape}")
print(f"  Labels shape    : {det['labels'].shape}")
print(f"  Scores shape    : {det['scores'].shape}")

if len(det["boxes"]) > 0:
    print(f"\n  First 3 detections:")
    for i in range(min(3, len(det["boxes"]))):
        box   = det["boxes"][i].tolist()
        label = det["labels"][i].item()
        score = det["scores"][i].item()
        print(f"    [{i}] box={[round(b,1) for b in box]}  "
              f"label={label}  score={score:.3f}")
else:
    print("\n  No detections (expected — random weights + random image)")

=== Inference Mode Output ===

  Keys in output  : ['boxes', 'labels', 'scores']
  Boxes shape     : torch.Size([100, 4])
  Labels shape    : torch.Size([100])
  Scores shape    : torch.Size([100])

  First 3 detections:
    [0] box=[353.2, 735.0, 400.1, 799.5]  label=3  score=0.173
    [1] box=[337.4, 733.9, 384.4, 799.5]  label=3  score=0.172
    [2] box=[656.4, 684.9, 702.6, 776.9]  label=3  score=0.172


In [5]:
# Inspect the anchor generator to confirm setup

rpn = model.rpn
ag  = rpn.anchor_generator

print("=== Anchor Configuration ===\n")
print(f"  Sizes per level  : {ag.sizes}")
print(f"  Aspect ratios    : {ag.aspect_ratios}")
print(f"  Anchors per pos  : {len(ag.sizes[0]) * len(ag.aspect_ratios[0])}")

# Calculate total anchors for a typical 800x800 image
feature_map_sizes = [200, 100, 50, 25, 13]   # H=W for 800x800 input
level_names       = ["P2","P3","P4","P5","P6"]

print(f"\n  {'Level':<6} {'FM Size':<10} {'Anchors':<12}")
print(f"  {'-'*28}")
total_anchors = 0
for name, fm_size in zip(level_names, feature_map_sizes):
    anchors = fm_size * fm_size * 3   # 3 aspect ratios
    total_anchors += anchors
    print(f"  {name:<6} {str(fm_size)+'x'+str(fm_size):<10} {anchors:<12,}")
print(f"  {'-'*28}")
print(f"  {'TOTAL':<16} {total_anchors:,}")
print(f"\n  RPN then filters these down to ~2000 proposals")

=== Anchor Configuration ===

  Sizes per level  : ((32,), (64,), (128,), (256,), (512,))
  Aspect ratios    : ((0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0), (0.5, 1.0, 2.0))
  Anchors per pos  : 3

  Level  FM Size    Anchors     
  ----------------------------
  P2     200x200    120,000     
  P3     100x100    30,000      
  P4     50x50      7,500       
  P5     25x25      1,875       
  P6     13x13      507         
  ----------------------------
  TOTAL            159,882

  RPN then filters these down to ~2000 proposals


In [6]:
checks = {}

# 1. Model builds
checks["Model builds without error"] = model is not None

# 2. Check all 4 losses present
model.train()
with torch.no_grad():
    loss_dict = model(dummy_images + dummy_images, dummy_targets + dummy_targets)

expected_losses = {"loss_classifier", "loss_box_reg",
                   "loss_objectness", "loss_rpn_box_reg"}
checks["All 4 losses present"] = expected_losses == set(loss_dict.keys())

# 3. Check inference output format
model.eval()
with torch.no_grad():
    dets = model([torch.rand(3, 800, 800).to(device)])
checks["Inference returns boxes/labels/scores"] = (
    "boxes" in dets[0] and "labels" in dets[0] and "scores" in dets[0]
)

# 4. Check head replaced correctly
in_feat  = model.roi_heads.box_predictor.cls_score.in_features
out_feat = model.roi_heads.box_predictor.cls_score.out_features
checks[f"Head has {NUM_CLASSES} output classes"] = out_feat == NUM_CLASSES

# 5. Check backbone frozen correctly
frozen = any(not p.requires_grad for p in model.backbone.parameters())
checks["Early backbone layers frozen"] = frozen

# 6. Device check
checks["Model on correct device"] = next(model.parameters()).device.type == device.type

print("=== Step 3 Checklist ===\n")
all_passed = True
for check, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {check}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All checks passed — ready for Step 4 (Loss Functions)!")
else:
    print("Fix the failing checks before moving on.")

=== Step 3 Checklist ===

  ✅  Model builds without error
  ✅  All 4 losses present
  ✅  Inference returns boxes/labels/scores
  ✅  Head has 7 output classes
  ✅  Early backbone layers frozen
  ✅  Model on correct device

All checks passed — ready for Step 4 (Loss Functions)!
